# Phân tích bổ sung từ kết quả đã có

Notebook này **không tải dữ liệu**. Nó đọc kết quả đã lưu của các notebook và script khác để trả lời năm câu hỏi nảy sinh khi review báo cáo:

- **A1.** Công thức S so với EPSS và CVSS ở từng khối lượng vá (thực nghiệm 4, `src/05_kiem_chung.py`).
- **A2.** Mô phỏng vận hành hằng tháng: thêm chỉ số vá trước khi bị khai thác.
- **A3.** Điểm S trước khi vào KEV, chia theo tuổi của lỗ hổng.
- **A4.** Nhóm vá khẩn cấp khi thêm quy tắc: lỗ hổng đã có trong KEV luôn được vá khẩn cấp.
- **A5.** Kiểm thử ngược đánh giá được bao nhiêu phần lỗ hổng sẽ vào KEV.

Đầu vào: `../output/kiem_chung.json`, `../output/kham_pha.json`, `ket_qua_S/so_lieu_S.json`, `ket_qua_truoc_kev/chi_tiet_tung_lo_hong.csv`. Kết quả lưu vào thư mục `ket_qua_bo_sung`.

## 1. Cấu hình và hàm dùng chung

In [1]:
import os, json
import numpy as np
import pandas as pd
from IPython.display import display

OUT_DIR = "ket_qua_bo_sung"
os.makedirs(OUT_DIR, exist_ok=True)
KC = json.load(open("../output/kiem_chung.json", encoding="utf-8"))        # thuc nghiem 4
KP = json.load(open("../output/kham_pha.json", encoding="utf-8"))          # phan tich kham pha cua thuc nghiem 4
S_JS = json.load(open("ket_qua_S/so_lieu_S.json", encoding="utf-8"))       # thuc nghiem 2
TRUOC = pd.read_csv("ket_qua_truoc_kev/chi_tiet_tung_lo_hong.csv")         # thuc nghiem 3
NGAY_DANH_GIA = ["2023-12-31", "2024-12-31", "2025-06-30"]
MUC_CAO = ["70 đến dưới 80", "80 đến dưới 90", "90 đến 100"]

def so(x): return f"{int(round(x)):,}".replace(",", ".")
def pct(v, k=2): return f"{v:.{k}f}".replace(".", ",")
def ty_le(a, b, k=2):
    """Ghi ro tu so va mau so. Tu so le (gia tri ky vong) thi lay 1 chu so thap phan."""
    if b == 0: return ""
    a_txt = so(a) if float(a).is_integer() else pct(a, 1)
    return f"{a_txt} / {so(b)} = {pct(100 * a / b, k)}%"

BANG, SO_LIEU = {}, {}

## 2. Bảng A1. Công thức S, EPSS và CVSS theo khối lượng vá

Thực nghiệm 4: lỗ hổng công bố năm 2024 và 2025 chia thành đợt theo tháng, EPSS lấy đúng ngày quyết định (cuối tháng công bố), nhãn là vào KEV trong 180 ngày sau. Mỗi cách xếp hạng chọn cùng một số lượng lỗ hổng có điểm cao nhất; lỗ hổng bằng điểm ở ranh giới được tính theo giá trị kỳ vọng.

Báo cáo trước chỉ so sánh với EPSS ở khối lượng 1% và 5%. Bảng này đặt cạnh nhau mọi khối lượng để thấy công thức S hơn hay kém EPSS ở đâu.

In [2]:
KL = [("0.01", "Vá 1%"), ("0.05", "Vá 5%"), ("0.1", "Vá 10%"), ("0.25", "Vá 25%"), ("W_csa", "Vá bằng số lượng của quy tắc CSA")]
PP = [("CVSS", "Theo CVSS"), ("EPSS", "Theo EPSS"), ("Cong thuc so mu 0.3", "Theo công thức S")]
rows, gop = [], {}
for y in ["2024", "2025"]:
    r = KC["rq1"][y]; P = r["P"]
    for f, ten in KL:
        h = {m: r["ket_qua"][m][f] for m, _ in PP}
        for m, _ in PP: gop[(f, m)] = gop.get((f, m), 0) + h[m]
        rows.append([y, ten, *[ty_le(h[m], P, 1) for m, _ in PP], pct(h["Cong thuc so mu 0.3"] - h["EPSS"], 1)])
P = KC["rq1"]["2024"]["P"] + KC["rq1"]["2025"]["P"]
SO_LIEU["a1_gop"] = {"P": P}
for f, ten in KL:
    h = {m: gop[(f, m)] for m, _ in PP}
    rows.append(["Gộp 2024 và 2025", ten, *[ty_le(h[m], P, 1) for m, _ in PP], pct(h["Cong thuc so mu 0.3"] - h["EPSS"], 1)])
    SO_LIEU["a1_gop"][ten] = {m: float(h[m]) for m, _ in PP}
a1 = pd.DataFrame(rows, columns=["Năm kiểm thử", "Khối lượng vá", *[t for _, t in PP], "S hơn EPSS (số ca)"])
BANG["A1 Theo khoi luong va"] = a1
display(a1)

h1 = KC["h1"]
print("Khoảng tin cậy 95% bootstrap của chênh lệch, gộp 2024 và 2025:")
for ten, d in [("Vá bằng số lượng của quy tắc CSA, S so với CVSS (H1)", h1["CVSS"]),
               ("Vá bằng số lượng của quy tắc CSA, S so với EPSS (H1)", h1["EPSS"])]:
    print(f"  {ten}: {pct(d['chenh_lech'], 1)} ca, từ {pct(d['ci'][0], 1)} đến {pct(d['ci'][1], 1)}")
for k, d in KP.items():
    f, m = k.split("_")
    print(f"  Vá {pct(100 * float(f), 0)}%, S so với {m} (khám phá): {pct(d['chenh'], 1)} ca, từ {pct(d['ci'][0], 1)} đến {pct(d['ci'][1], 1)}")

,Năm kiểm thử,Khối lượng vá,Theo CVSS,Theo EPSS,Theo công thức S,S hơn EPSS (số ca)
0,2024,Vá 1%,"0,9 / 50 = 1,8%","7 / 50 = 14,0%","7 / 50 = 14,0%","0,0"
1,2024,Vá 5%,"9,7 / 50 = 19,4%","9,6 / 50 = 19,2%","10 / 50 = 20,0%","0,4"
2,2024,Vá 10%,"20,7 / 50 = 41,4%","11 / 50 = 22,0%","13,7 / 50 = 27,4%","2,7"
3,2024,Vá 25%,"27 / 50 = 54,0%","14,7 / 50 = 29,4%","29 / 50 = 58,0%","14,3"
4,2024,Vá bằng số lượng của quy tắc CSA,"27 / 50 = 54,0%","14,8 / 50 = 29,6%","29 / 50 = 58,0%","14,2"
5,2025,Vá 1%,"6,5 / 66 = 9,9%","14 / 66 = 21,2%","13 / 66 = 19,7%","-1,0"
6,2025,Vá 5%,"12,1 / 66 = 18,3%","26 / 66 = 39,4%","30 / 66 = 45,5%","4,0"
7,2025,Vá 10%,"19,0 / 66 = 28,8%","36 / 66 = 54,5%","35 / 66 = 53,0%","-1,0"
8,2025,Vá 25%,"39,1 / 66 = 59,2%","42 / 66 = 63,6%","43 / 66 = 65,2%","1,0"
9,2025,Vá bằng số lượng của quy tắc CSA,"39 / 66 = 59,1%","42 / 66 = 63,6%","43 / 66 = 65,2%","1,0"


Khoảng tin cậy 95% bootstrap của chênh lệch, gộp 2024 và 2025:
  Vá bằng số lượng của quy tắc CSA, S so với CVSS (H1): 6,0 ca, từ -2,0 đến 15,0
  Vá bằng số lượng của quy tắc CSA, S so với EPSS (H1): 15,2 ca, từ 5,2 đến 26,7
  Vá 1%, S so với CVSS (khám phá): 12,6 ca, từ 3,9 đến 21,8
  Vá 1%, S so với EPSS (khám phá): -1,0 ca, từ -3,0 đến 2,0
  Vá 5%, S so với CVSS (khám phá): 18,2 ca, từ 8,3 đến 28,5
  Vá 5%, S so với EPSS (khám phá): 4,4 ca, từ -0,7 đến 10,5


## 3. Bảng A2. Mô phỏng vận hành hằng tháng

Mỗi cuối tháng, mỗi cách chọn vá một số lỗ hổng có điểm cao nhất trong các lỗ hổng chưa vá, với cùng ngân sách. Hai chỉ số khác nhau về ý nghĩa:

- **Vá trước khi bị khai thác:** đo khả năng chủ động, tức dự báo đúng trước khi lỗ hổng vào KEV.
- **An toàn trong 30 ngày:** vá trước, hoặc chậm nhất 30 ngày sau khi vào KEV. Chỉ số này phụ thuộc nhiều vào việc phản ứng sau khi lỗ hổng vào KEV.

Cây VMC ở đây là biến thể do nhóm cài đặt: tách lỗ hổng trong KEV và lỗ hổng có EPSS từ 0,088 thành hai tầng, còn bài báo gốc gộp hai điều kiện này vào một bước lọc.

In [3]:
SIM = [("CVSS", "CVSS"), ("EPSS", "EPSS"), ("Cong thuc bo KEV", "Công thức bỏ KEV"),
       ("Cong thuc day du", "Công thức đầy đủ"), ("Cay Shimizu", "Cây VMC (biến thể của nhóm)")]
rows = []
for y in ["2024", "2025"]:
    for b in ["0.1", "0.25", "0.5"]:
        for m, ten in SIM:
            s = KC["rq2"][y][b][m]
            rows.append([y, f"{pct(100 * float(b), 0)}%", ten, f"{pct(100 * s['va_truoc'], 1)}%", f"{pct(100 * s['an_toan_30'], 1)}%",
                         "" if s["tre_trung_vi"] is None else f"{s['tre_trung_vi']:.0f} ngày", f"{pct(100 * s['chua_va'], 1)}%"])
a2 = pd.DataFrame(rows, columns=["Năm", "Ngân sách vá", "Cách chọn", "Vá trước khi bị khai thác",
                                 "An toàn trong 30 ngày", "Trễ trung vị", "Chưa vá đến cuối kỳ"])
BANG["A2 Mo phong"] = a2
display(a2[a2["Ngân sách vá"] == "25%"])

,Năm,Ngân sách vá,Cách chọn,Vá trước khi bị khai thác,An toàn trong 30 ngày,Trễ trung vị,Chưa vá đến cuối kỳ
5,2024,25%,CVSS,"20,3%","55,5%",14 ngày,"42,2%"
6,2024,25%,EPSS,"16,4%","93,8%",16 ngày,"3,9%"
7,2024,25%,Công thức bỏ KEV,"23,4%","91,4%",16 ngày,"5,5%"
8,2024,25%,Công thức đầy đủ,"23,4%","100,0%",16 ngày,"0,0%"
9,2024,25%,Cây VMC (biến thể của nhóm),"22,7%","100,0%",16 ngày,"0,0%"
20,2025,25%,CVSS,"19,2%","56,4%",15 ngày,"37,2%"
21,2025,25%,EPSS,"24,4%","98,3%",17 ngày,"0,6%"
22,2025,25%,Công thức bỏ KEV,"26,7%","98,3%",17 ngày,"1,7%"
23,2025,25%,Công thức đầy đủ,"26,7%","100,0%",17 ngày,"0,0%"
24,2025,25%,Cây VMC (biến thể của nhóm),"20,3%","100,0%",17 ngày,"0,0%"


## 4. Bảng A3. Điểm S trước khi vào KEV, chia theo tuổi của lỗ hổng

CISA chỉ đưa lỗ hổng vào KEV **sau khi** có bằng chứng bị khai thác. Vì vậy điểm S một ngày trước khi vào KEV đo khả năng phát hiện trước khi CISA xác nhận, chưa phải dự báo trước khi bị khai thác.

Bảng chia 836 lỗ hổng vào KEV từ 01/04/2023 theo khoảng thời gian từ ngày công bố đến ngày vào KEV, để xem điểm S cao đến từ lỗ hổng mới công bố hay lỗ hổng cũ.

In [4]:
t = TRUOC.copy()
for c in ["ngay_kev", "ngay_cong_bo"]: t[c] = pd.to_datetime(t[c])
t["tuoi"] = (t["ngay_kev"] - t["ngay_cong_bo"]).dt.days
co = t[t["epss_1"].notna()]
NHOM_TUOI = [("Công bố tối đa 30 ngày trước khi vào KEV", -10**6, 30),
             ("Công bố 31 đến 365 ngày trước khi vào KEV", 31, 365),
             ("Công bố hơn 1 năm trước khi vào KEV", 366, 10**6)]
rows = [["Chưa có điểm EPSS 1 ngày trước khi vào KEV", so(t["epss_1"].isna().sum()), "", "", ""]]
for ten, a, b in NHOM_TUOI + [("Tất cả lỗ hổng có điểm EPSS", -10**6, 10**6)]:
    g = co[co["tuoi"].between(a, b)]
    rows.append([ten, so(len(g)), pct(g["epss_1"].median(), 4),
                 ty_le(int((g["S_1"] >= 70).sum()), len(g)), ty_le(int((g["S_1"] >= 50).sum()), len(g))])
a3 = pd.DataFrame(rows, columns=["Nhóm lỗ hổng KEV", "Số lượng", "Trung vị EPSS 1 ngày trước khi vào KEV",
                                 "S 1 ngày trước từ 70", "S 1 ngày trước từ 50"])
BANG["A3 Truoc KEV theo tuoi"] = a3
display(a3)
cao = co[co["S_1"] >= 70]
SO_LIEU["a3"] = dict(trung_vi_epss_1=float(co["epss_1"].median()), so_S70=len(cao),
                     S70_cu_hon_1_nam=int((cao["tuoi"] > 365).sum()),
                     nhom={ten: dict(n=int(co["tuoi"].between(a, b).sum()),
                                     S70=int((co.loc[co["tuoi"].between(a, b), "S_1"] >= 70).sum()),
                                     trung_vi_epss=float(co.loc[co["tuoi"].between(a, b), "epss_1"].median()))
                           for ten, a, b in NHOM_TUOI})
print(f"Trong {len(cao)} lỗ hổng có S từ 70 một ngày trước khi vào KEV: "
      f"{ty_le(int((cao['tuoi'] > 365).sum()), len(cao))} đã công bố hơn 1 năm trước khi vào KEV")

,Nhóm lỗ hổng KEV,Số lượng,Trung vị EPSS 1 ngày trước khi vào KEV,S 1 ngày trước từ 70,S 1 ngày trước từ 50
0,Chưa có điểm EPSS 1 ngày trước khi vào KEV,227,,,
1,Công bố tối đa 30 ngày trước khi vào KEV,224,"0,0009","10 / 224 = 4,46%","16 / 224 = 7,14%"
2,Công bố 31 đến 365 ngày trước khi vào KEV,196,"0,0039","30 / 196 = 15,31%","45 / 196 = 22,96%"
3,Công bố hơn 1 năm trước khi vào KEV,189,"0,0709","68 / 189 = 35,98%","86 / 189 = 45,50%"
4,Tất cả lỗ hổng có điểm EPSS,609,"0,0038","108 / 609 = 17,73%","147 / 609 = 24,14%"


Trong 108 lỗ hổng có S từ 70 một ngày trước khi vào KEV: 68 / 108 = 62,96% đã công bố hơn 1 năm trước khi vào KEV


## 5. Bảng A4. Nhóm vá khẩn cấp khi thêm quy tắc KEV

Ở nhánh KEV, S = CVSS × 10, nên lỗ hổng đã bị khai thác nhưng có CVSS thấp vẫn có thể nằm dưới ngưỡng vá khẩn cấp. Quy tắc bổ sung: **mọi lỗ hổng đã có trong KEV luôn thuộc nhóm vá khẩn cấp**, giống cách làm của Chỉ thị BOD 22-01.

Bảng tính khối lượng nhóm vá khẩn cấp khi dùng ngưỡng S từ 70 cùng quy tắc này, và hiệu suất của ngưỡng S từ 70 so với quy tắc CSA trên phần chưa có trong KEV. Hiệu suất là tỷ lệ bắt được chia cho tỷ lệ khối lượng vá, tức gấp bao nhiêu lần chọn ngẫu nhiên.

In [5]:
rows = []
SO_LIEU["a4"] = {}
for T in NGAY_DANH_GIA:
    x = S_JS[T]; muc = x["muc"]
    n70, k70 = sum(muc[m]["n"] for m in MUC_CAO), sum(muc[m]["k"] for m in MUC_CAO)
    hs70 = (k70 / x["P"]) / (n70 / x["chua"]); hscsa = (x["csa"]["k"] / x["P"]) / (x["csa"]["n"] / x["chua"])
    rows.append([T, so(x["N"]), so(n70), so(x["kev_T"]), ty_le(n70 + x["kev_T"], x["N"]),
                 ty_le(x["csa"]["n"], x["chua"]), ty_le(k70, x["P"]), f"{pct(hs70, 1)} lần", f"{pct(hscsa, 1)} lần"])
    SO_LIEU["a4"][T] = dict(n70=n70, k70=k70, kev_T=x["kev_T"], N=x["N"], hieu_suat_S70=hs70, hieu_suat_csa=hscsa)
a4 = pd.DataFrame(rows, columns=["Ngày đánh giá", "Lỗ hổng có CVSS 3.x", "Chưa có KEV, S từ 70", "Đã có trong KEV",
                                 "Nhóm khẩn cấp (S từ 70 hoặc có trong KEV)", "Quy tắc CSA (phần chưa có KEV)",
                                 "S từ 70 bắt được", "S từ 70: gấp ngẫu nhiên", "Quy tắc CSA: gấp ngẫu nhiên"])
BANG["A4 Nhom khan cap"] = a4
display(a4)
km = S_JS["kev_muc"]
duoi = sum(v for m, v in km.items() if int(m.split()[0]) < 70)
SO_LIEU["a4_kev_duoi_70"] = dict(so=duoi, tong=sum(km.values()))
print(f"Ngày 2025-06-30, nếu chỉ dùng ngưỡng S từ 70: {ty_le(duoi, sum(km.values()))} lỗ hổng đã có trong KEV "
      f"nằm ngoài nhóm vá khẩn cấp")

,Ngày đánh giá,Lỗ hổng có CVSS 3.x,"Chưa có KEV, S từ 70",Đã có trong KEV,Nhóm khẩn cấp (S từ 70 hoặc có trong KEV),Quy tắc CSA (phần chưa có KEV),S từ 70 bắt được,S từ 70: gấp ngẫu nhiên,Quy tắc CSA: gấp ngẫu nhiên
0,2023-12-31,148.366,1.499,1.053,"2.552 / 148.366 = 1,72%","42.685 / 147.313 = 28,98%","14 / 61 = 22,95%","22,6 lần","2,1 lần"
1,2024-12-31,187.995,1.464,1.239,"2.703 / 187.995 = 1,44%","52.662 / 186.756 = 28,20%","16 / 80 = 20,00%","25,5 lần","2,1 lần"
2,2025-06-30,210.994,2.803,1.371,"4.174 / 210.994 = 1,98%","58.133 / 209.623 = 27,73%","34 / 91 = 37,36%","27,9 lần","2,1 lần"


Ngày 2025-06-30, nếu chỉ dùng ngưỡng S từ 70: 160 / 1.371 = 11,67% lỗ hổng đã có trong KEV nằm ngoài nhóm vá khẩn cấp


## 6. Bảng A5. Phạm vi của kiểm thử ngược

Tại mỗi ngày đánh giá, công thức chỉ chấm điểm được lỗ hổng **đã công bố**. Bảng cho biết trong số lỗ hổng vào KEV trong 12 tháng sau, bao nhiêu phần đã công bố trước ngày đánh giá (thuộc phạm vi kiểm thử) và bao nhiêu phần công bố sau. Phần công bố sau được thực nghiệm 4 xử lý, vì thực nghiệm này chấm điểm lỗ hổng mới theo từng tháng.

In [6]:
rows = []
for T in NGAY_DANH_GIA:
    x = S_JS[T]
    rows.append([T, so(x["kev_moi"]), ty_le(x["P"], x["kev_moi"]), ty_le(x["sau_T"], x["kev_moi"]),
                 ty_le(x["khong_cvss3"], x["kev_moi"])])
a5 = pd.DataFrame(rows, columns=["Ngày đánh giá", "Lỗ hổng vào KEV trong 12 tháng sau",
                                 "Đã công bố trước ngày đánh giá (được kiểm thử)", "Công bố sau ngày đánh giá",
                                 "Công bố trước nhưng không có CVSS 3.x"])
BANG["A5 Pham vi kiem thu"] = a5
display(a5)

,Ngày đánh giá,Lỗ hổng vào KEV trong 12 tháng sau,Đã công bố trước ngày đánh giá (được kiểm thử),Công bố sau ngày đánh giá,Công bố trước nhưng không có CVSS 3.x
0,2023-12-31,186,"61 / 186 = 32,80%","125 / 186 = 67,20%","0 / 186 = 0,00%"
1,2024-12-31,245,"80 / 245 = 32,65%","165 / 245 = 67,35%","0 / 245 = 0,00%"
2,2025-06-30,259,"91 / 259 = 35,14%","168 / 259 = 64,86%","0 / 259 = 0,00%"


## 7. Lưu kết quả

In [7]:
with pd.ExcelWriter(os.path.join(OUT_DIR, "bang_bo_sung.xlsx")) as w:
    for ten, b in BANG.items():
        b.to_excel(w, sheet_name=ten[:31], index=False)
for ten, b in BANG.items():
    b.to_csv(os.path.join(OUT_DIR, ten.replace(" ", "_") + ".csv"), index=False, encoding="utf-8-sig")
json.dump(SO_LIEU, open(os.path.join(OUT_DIR, "so_lieu_bo_sung.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2,
          default=lambda o: int(o) if isinstance(o, np.integer) else float(o))
print("Đã lưu:", ", ".join(sorted(os.listdir(OUT_DIR))))

Đã lưu:

 A1_Theo_khoi_luong_va.csv, A2_Mo_phong.csv, A3_Truoc_KEV_theo_tuoi.csv, A4_Nhom_khan_cap.csv, A5_Pham_vi_kiem_thu.csv, bang_bo_sung.xlsx, so_lieu_bo_sung.json
